# FaceGen — CVAE Training on CelebA
**Conditional VAE** : génère des visages en choisissant 40 attributs physiques.

Workflow :
1. GPU check
2. Installer les dépendances
3. Monter Drive + copier les données
4. Cloner le repo GitHub
5. Login wandb
6. Entraîner
7. Générer / manipuler des visages

## 1 · GPU check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

## 2 · Install dependencies

In [ ]:
# Colab ships with PyTorch — we only need Lightning, wandb, kaggle
!pip install -q pytorch-lightning wandb kaggle

## 3 · Kaggle — credentials & download dataset

In [ ]:
# Kaggle credentials via Colab Secrets (icône 🔑 dans le panneau gauche)
# Ajoute deux secrets : KAGGLE_USERNAME et KAGGLE_KEY
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

In [ ]:
import os, zipfile

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download (~1.7 GB)
!kaggle datasets download -d jessicali9530/celeba-dataset -p {DATA_DIR}

# Extract main zip
print("Extracting…")
with zipfile.ZipFile(f"{DATA_DIR}/celeba-dataset.zip") as z:
    z.extractall(DATA_DIR)
os.remove(f"{DATA_DIR}/celeba-dataset.zip")

# Les images sont dans un zip imbriqué sur ce dataset Kaggle
nested = f"{DATA_DIR}/img_align_celeba.zip"
if os.path.exists(nested):
    with zipfile.ZipFile(nested) as z:
        z.extractall(DATA_DIR)
    os.remove(nested)

# Sanity check
img_dir = f"{DATA_DIR}/img_align_celeba/img_align_celeba"
n = len(os.listdir(img_dir))
print(f"Images : {n:,}")
assert n == 202_599, f"Structure inattendue — vérifie {DATA_DIR}"

## 3b · Google Drive — checkpoints uniquement

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = "/content/drive/MyDrive/FaceGen/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints → {CKPT_DIR}")

## 4 · Clone repo

In [ ]:
REPO_URL = "https://github.com/JulesV19/face-gen.git"
REPO_DIR = "/content/face-gen"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## 5 · Weights & Biases login

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

## 6 · Training

In [ ]:
# L4 · 24 GB VRAM · bf16-mixed natif (Ada Lovelace)
# batch_size=128 @ 64×64 ≈ 3-4 GB VRAM → confortable
!python train.py \
    --data_dir {DATA_DIR} \
    --batch_size 128 \
    --max_epochs 100 \
    --lr 1e-3 \
    --latent_dim 256 \
    --beta_max 1.0 \
    --warmup_epochs 10 \
    --precision bf16-mixed \
    --num_workers 4 \
    --ckpt_dir {CKPT_DIR} \
    --wandb_project facegen-cvae

## 7 · Inference
### 7a · Génération pure — choisir les attributs

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from generate import load_model, build_condition, save_grid
from utils import show_grid, list_attrs

# ── EDIT: path to your best checkpoint ───────────────────────────────────────
CKPT = "/content/drive/MyDrive/FaceGen/checkpoints/best.ckpt"

device = "cuda"
model, cfg = load_model(CKPT, device)

print("All available attributes:")
list_attrs()

In [ ]:
# ── Customise here ────────────────────────────────────────────────────────────
# 1.0 = attribute present, 0.0 = absent, 0.5 = neutral
wanted_attrs = {
    "Smiling":      1.0,
    "Blond_Hair":   1.0,
    "Young":        1.0,
    "Wearing_Lipstick": 1.0,
    "High_Cheekbones":  1.0,
}

c = build_condition(wanted_attrs, cfg.num_attrs, default=0.0, device=device)
imgs = model.generate(c, n=16)

show_grid(imgs, nrow=8, title="Generated faces")

### 7b · Manipulation d'attributs sur une vraie photo

In [ ]:
from generate import load_image

# ── EDIT ─────────────────────────────────────────────────────────────────────
INPUT_IMG = "/content/data/img_align_celeba/img_align_celeba/000042.jpg"

x = load_image(INPUT_IMG, cfg.img_size, device)

# Source: provide the known attributes of the input image
# (or leave empty to use neutral 0.5 defaults)
c_src_dict = {"Smiling": 0.0, "Young": 1.0}
c_src = build_condition(c_src_dict, cfg.num_attrs, default=0.5, device=device)

# Target: flip Smiling on → generates the same person smiling
c_tgt = c_src.clone()
c_tgt[31] = 1.0  # index 31 = Smiling

manipulated = model.manipulate(x, c_src.unsqueeze(0), c_tgt.unsqueeze(0))

show_grid(
    torch.cat([x, manipulated]),
    nrow=2,
    title="Original | Smiling=1",
    figsize=(5, 4),
)

### 7c · Interpolation entre deux visages

In [ ]:
from utils import interpolate

IMG_A = "/content/data/img_align_celeba/img_align_celeba/000001.jpg"
IMG_B = "/content/data/img_align_celeba/img_align_celeba/000100.jpg"

c_neutral = build_condition({}, cfg.num_attrs, default=0.5, device=device)

x_a = load_image(IMG_A, cfg.img_size, device)
x_b = load_image(IMG_B, cfg.img_size, device)

z_a = model.encode_mu(x_a, c_neutral.unsqueeze(0))
z_b = model.encode_mu(x_b, c_neutral.unsqueeze(0))

frames = interpolate(model, z_a[0], z_b[0], c_neutral, steps=10)
show_grid(frames, nrow=10, title="Latent space interpolation", figsize=(16, 3))